In [1]:
Sys.setenv(TZ = "America/Vancouver")

In [2]:
install.packages("suncalc")

Installing package into 'C:/Users/Kaiyan Zhang/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



package 'suncalc' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Kaiyan Zhang\AppData\Local\Temp\Rtmpc3eYdh\downloaded_packages


In [3]:
library(sf)
library(dplyr)
library(lubridate)
library(suncalc)
library(readr)
library(purrr)


Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.3.1; sf_use_s2() is TRUE

Warning message:
"package 'dplyr' was built under R version 4.4.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union


Warning message:
"package 'suncalc' was built under R version 4.4.3"


In [4]:
# divisions <- st_read("census_divison.geojson", quiet = TRUE)
# divisions_valid <- st_make_valid(divisions)
# sf_use_s2(FALSE)
# vancouver_union <- st_union(divisions_valid)
# st_write(vancouver_union, "vancouver_union.geojson", delete_dsn = TRUE)


In [ ]:
Sys.setenv(TZ = "America/Vancouver")

# vancouver_union <- st_read("vancouver_union.geojson", quiet = TRUE)



crime_data_all <- read_csv("../crimedata_csv_AllNeighbourhoods_AllYears/crimedata_csv_AllNeighbourhoods_AllYears.csv")

crime_data_clean <- crime_data_all |>
  filter(!is.na(X), !is.na(Y)) |>
  filter((X != 0.0) & (Y != 0.0))|>
  filter(MONTH %in% c(1, 3, 5, 7, 9, 11))|>
  filter(!TYPE %in% c("Vehicle Collision or Pedestrian Struck (with Fatality)", "Vehicle Collision or Pedestrian Struck (with Injury)"))

crime_sf <- st_as_sf(crime_data_clean, coords = c("X", "Y"), crs = 26910)

crime_sf <- st_transform(crime_sf, 4326)

# crime_with_division <- st_join(crime_sf, divisions["name"], join = st_within)

# crime_in_vancouver <- crime_with_division[
#   st_within(crime_with_division, vancouver_union, sparse = FALSE),
# ]

crime_sf$crime_datetime <- as.POSIXct(
  paste(crime_sf$YEAR,
        crime_sf$MONTH,
        crime_sf$DAY,
        crime_sf$HOUR,
        crime_sf$MINUTE,
        sep = "-"),
  format = "%Y-%m-%d-%H-%M",
  tz = "America/Vancouver"
)

crime_sf$date_only <- as.Date(crime_sf$crime_datetime)

unique_dates <- unique(crime_sf$date_only)
sun_times <- getSunlightTimes(
  date = unique_dates,
  lat = 49.2827,
  lon = -123.1207,
  keep = c("sunrise", "sunset"),
  tz = "America/Vancouver"
) %>%
  select(date, sunrise, sunset)

crime_sf <- left_join(crime_sf, sun_times, 
                                by = c("date_only" = "date"))

crime_sf$time_category <- ifelse(
  crime_sf$crime_datetime >= crime_sf$sunrise &
    crime_sf$crime_datetime < crime_sf$sunset,
  "day",
  "night"
)

# crime_sf <- st_set_geometry(crime_sf, NULL)

write_csv(crime_sf, "crime_in_vancouver.csv")

crime_day <- crime_sf %>% filter(time_category == "day")
write_csv(st_set_geometry(crime_day, NULL), "crime_day.csv")

crime_night <- crime_sf %>% filter(time_category == "night")
write_csv(st_set_geometry(crime_night, NULL), "crime_night.csv")

cat("处理完成：\n",
    "1) crime_in_vancouver_with_division.csv\n",
    "2) crime_day.csv\n",
    "3) crime_night.csv\n")


Rows: 910710 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


处理完成：
 1) crime_in_vancouver_with_division.csv
 2) crime_day.csv
 3) crime_night.csv
